[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR/blob/main/colab_dae_h.ipynb)

# Part A: the size of h and dropout

`colab_dae_full.ipynb` chose salt and pepper 0.5 with dropout 0.3 on h (val k-NN 0.1058). Its val k-NN still rose with dropout (0, 0.1, 0.2, 0.3: 0.0990, 0.1019, 0.1045, 0.1058), so here dropout goes higher.

h is 256, 2.1% of the input. In the course autoencoder (MNIST) h is 64: 8.2% of the input, 2.0% of the flattened conv output. The same shares here are 1,024 and about 384.

- **h** 256, 384, 1,024 × **dropout** 0.3, 0.4, 0.5: 9 runs on all the images for 20 epochs, by **val k-NN top-1** (the best epoch is kept). h 256 with dropout 0.3 is the choice of `colab_dae_full.ipynb`, read from Drive: 8 new runs.
- About 12 minutes a run on an L4, 1.6 hours in all. `PART` splits the runs between two runtimes (about 50 minutes each).
- The last cell saves the best as `"dae"` in `tuning_choice.json`; the choice of `colab_dae_full.ipynb` is kept as `"dae_full20"`.
- The decoder stays the mirror of the encoder (light).

## Setup
Downloads the repository (or pulls the latest code), installs it and checks the GPU.

In [ ]:
import os

REPO = "/content/Representation-Learning-Midterm-DAE-SimCLR"
if not os.path.exists(REPO):
    !git clone https://github.com/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR.git {REPO}
os.chdir(REPO)
!git pull
!git log --oneline -1

In [ ]:
%pip install -q -r requirements.txt

import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → L4 GPU, then rerun this cell."
print(torch.cuda.get_device_name(0), "|", os.cpu_count(), "CPU cores")

## Drive
Every run writes to `midterm_runs`, next to the earlier runs. The runs here have `tune_pretrain_DAE` in their folder name.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/לימודים/תואר שני/שנה ב/סמסטר א/למידת ייצוגים במודלים עמוקים/מטלות/מטלת אמצע"
OUT = f"{DRIVE_DIR}/midterm_runs"
assert os.path.isdir(OUT), f"no folder {OUT}"

## TensorBoard
Filter the runs with `tune_pretrain`, then *Toggle all runs*. `epoch/val_knn_top1` shows where each run peaks, `epoch/val_loss` the reconstruction, the TEXT tab each run's `run_info`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{OUT}"

## The runs
`train` runs train.py only if the run has no `results.json` yet: after a disconnect a finished run is read from Drive and an unfinished one continues from its `last.ckpt`.

`PART`: 0 runs all 9 here. To use two runtimes, set 1 in one and 2 in the other (every second run each), and run the last cell when both are done.

In [ ]:
import importlib
import itertools
import json
import sys

import pandas as pd

RUN = "--mode pretrain --tune --num_epochs 20"  # all the images, the full k-NN bank, val only (no test)
CHOICE_FILE = f"{OUT}/tuning_choice.json"
# The choice of colab_dae_full.ipynb (salt and pepper 0.5, dropout 0.3, val k-NN 0.1058 in 20 epochs)
base = {"noise_type": "salt_pepper", "noise_level": 0.5, "patch_size": 8, "conv1": 32, "conv2": 64,
        "latent_dim": 256, "dropout": 0.3, "augment": "flip", "batch_size": 512, "optimizer": "adamw",
        "activation": "gelu", "lr": 0.001, "wd": 0.01, "scheduler": "none", "noise_type2": "none",
        "noise_level2": 0.0}


def flags_of(params):
    """The train.py flags of one DAE."""
    p = dict(params)
    return f"--channels {p.pop('conv1')} {p.pop('conv2')} " + " ".join(f"--{k} {v}" for k, v in p.items())


def folder_of(flags):
    """The run folder train.py uses for these flags (run_info.folder_name)."""
    import config, utils, run_info
    for module in (config, utils, run_info):
        importlib.reload(module)
    sys.argv = ["train.py", *flags.split(), "--output_dir", OUT]
    return f"{OUT}/{run_info.folder_name(utils.parse_args().mode)}"


def train(flags):
    """One run of train.py (skipped if it finished before): its val numbers and its folder."""
    if not os.path.exists(f"{folder_of(flags)}/results.json"):
        !python train.py {flags} --output_dir "{OUT}"
    folder = folder_of(flags)
    with open(f"{folder}/results.json") as f:
        return {**json.load(f), "run": os.path.basename(folder)}

## h × dropout
Every combination, the rest as in `base`: 3 × 3 = 9 runs, 20 epochs each.

In [ ]:
PART = 0  # 0: all 9 runs; 1 or 2: every second run, for two runtimes
grid = list(itertools.product([256, 384, 1024], [0.3, 0.4, 0.5]))
rows = []
for latent_dim, dropout in grid[PART - 1::2] if PART else grid:
    params = {**base, "latent_dim": latent_dim, "dropout": dropout}
    r = train(f"{RUN} {flags_of(params)}")
    rows.append({**params, "val_knn": r["val_knn_top1"], "val_loss": r["val_loss"], "run": r["run"]})
display(pd.DataFrame(rows).sort_values("val_knn", ascending=False)[["latent_dim", "dropout", "val_knn", "val_loss"]].round(4))

## The choice
All 9 runs, read from Drive (with two runtimes, both must be done). The best by val k-NN is saved as `"dae"`.

In [ ]:
choices = json.load(open(CHOICE_FILE))
choices.setdefault("dae_full20", choices["dae"])  # the choice of colab_dae_full.ipynb, kept
finals = []
for h, d in grid:
    folder = folder_of(f"{RUN} {flags_of({**base, 'latent_dim': h, 'dropout': d})}")
    assert os.path.exists(f"{folder}/results.json"), f"not finished yet: {os.path.basename(folder)}"
    r = json.load(open(f"{folder}/results.json"))
    finals.append({**base, "latent_dim": h, "dropout": d, "val_knn": r["val_knn_top1"], "val_loss": r["val_loss"],
                   "run": os.path.basename(folder)})
finals = pd.DataFrame(finals).sort_values("val_knn", ascending=False)
display(finals[["latent_dim", "dropout", "val_knn", "val_loss", "run"]].round(4))

best_row = finals.iloc[0]
choice = {**{k: best_row[k] for k in base}, "num_epochs": 20}
choice = {k: v.item() if hasattr(v, "item") else v for k, v in choice.items()}  # numpy -> python for json
choices["dae"] = choice
with open(CHOICE_FILE, "w") as f:
    json.dump(choices, f, indent=1)
print("the DAE:", choice)